In [1]:
import numpy as np
import pandas as pd

In [3]:
data = {
    "Age":[22,25,47,52,46,56] ,
    "Salary": [15000,29000,48000,60000,52000,62000] ,
    "Buy": ["No","No","Yes","Yes","Yes","Yes"]
}

df = pd.DataFrame(data)
df

,Age,Salary,Buy
0,22,15000,No
1,25,29000,No
2,47,48000,Yes
3,52,60000,Yes
4,46,52000,Yes
5,56,62000,Yes


In [4]:
df["Buy"] = df["Buy"].map({"No":0,"Yes":1})

In [5]:
X = df[["Age","Salary"]].values
y = df["Buy"].values

In [6]:
print(X)

[[   22 15000]
 [   25 29000]
 [   47 48000]
 [   52 60000]
 [   46 52000]
 [   56 62000]]


In [7]:
print(y)

[0 0 1 1 1 1]


In [ ]:
def entropy(y):
    # finds unique classes and how many times they occur. 
    # y = [0,0,1,1,1,1] so -> classes = [0,1] and counts = [2,4]
    classes , counts = np.unique(y,return_counts=True)

    # calculates class probabilities
    probabilities = counts/counts.sum()

    # formula of entropy 
    entropy_value = -np.sum(probabilities*np.log2(probabilities))

    return entropy_value

In [ ]:
# testing that entropy function giving y as input that we have in dataset
entropy(y)

0.9182958340544896

In [12]:
def information_gain(parent,left,right):

    parent_entropy = entropy(parent)

    n = len(parent)
    n_left = len(left)
    n_right = len(right)

    child_entropy = (n_left/n)*entropy(left) + (n_right/n)*entropy(right)

    ig = parent_entropy - child_entropy

    return ig


In [19]:
def best_split(X,Y):

    best_feature = None
    best_threshold = None
    best_gain = -1

    n_features = X.shape[1]

    for feature in range(n_features):
        thresholds = np.unique(X[:,feature])

        for threshold in thresholds:

            left_indices = X[:,feature] <= threshold

            right_indices = X[:,feature] > threshold
            
            left_y = y[left_indices]
            right_y = y[right_indices]

            if len(left_y) == 0 or len(right_y) == 0:
                continue

            gain  = information_gain(y,left_y,right_y)

            if gain > best_gain:

                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature , best_threshold 


In [21]:
feature , threshold = best_split(X,y)

print("Best feature: ",feature)
print("Best threshold: ",threshold)
# print("information_gain: ",gain)


Best feature:  0
Best threshold:  25


In [22]:
class Node:
    def __init__(self,feature = None,threshold=None,left=None,right=None,value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        

In [25]:
def build_tree(X,y,depth=0,max_depth=3):
    if len(set(y))==1:
        return Node(value=y[0])
    
    if depth >=max_depth:
        return Node(value=np.bincount(y).argmax())
    
    feature,threshold = best_split(X,y)

    if feature is None:
        return Node(value=np.bincount(y).argmax())
    
    left_indices = X[:,feature] <=threshold
    right_indices = X[:,feature] > threshold

    left_child = build_tree(X[left_indices],y[left_indices],depth+1,max_depth)
    right_child = build_tree(X[right_indices],y[right_indices],depth+1,max_depth)

    return Node(feature,threshold,left_child,right_child)
    

In [26]:
tree = build_tree(X,y,max_depth=3)

In [27]:
def predict_sample(x,node):
    if node.value is not None:
        return node.value
    
    if x[node.feature] <= node.threshold:
        return predict_sample(x,node.left)
    else:
        return predict_sample(x,node.right)

In [28]:
def predict(X,tree):
    predictions = []

    for x in X:
        predictions.append(predict_sample(x,tree))
    
    return np.array(predictions)

In [29]:
preds = predict(X,tree)

print("Predictions:",preds)
print("Actual:",y)

Predictions: [0 0 1 1 1 1]
Actual: [0 0 1 1 1 1]


In [30]:
def print_tree(node, depth=0):

    # indentation to show tree structure
    indent = "  " * depth

    # if it is a leaf node
    if node.value is not None:
        print(indent + "Leaf:", node.value)
        return

    # print decision rule
    print(indent + f"Feature {node.feature} <= {node.threshold}")

    # left branch
    print(indent + "Left:")
    print_tree(node.left, depth + 1)

    # right branch
    print(indent + "Right:")
    print_tree(node.right, depth + 1)

In [31]:
print_tree(tree)

Feature 0 <= 25
Left:
  Leaf: 0
Right:
  Leaf: 1
